<a href="https://colab.research.google.com/github/jumafernandez/ANN-UNSL/blob/main/notebooks/notebook_01_carga_datos_y_embeddings.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🧩 Notebook 1 — Corpus MultiWOZ 2.2 y generación de embeddings

En esta notebook construimos el pipeline completo desde el dataset crudo de diálogos hasta una matriz de embeddings lista para usar en algoritmos de *Approximate Nearest Neighbors* (ANN).

Trabajamos con el corpus **MultiWOZ 2.2**, un dataset clásico de *Task-Oriented Dialogue (TOD)* en inglés. A partir de este corpus:

1. Descargamos y exploramos la estructura de archivos del dataset.
2. Unificamos los splits `train`, `dev` y `test` en un único DataFrame de turnos.
3. Generamos embeddings semánticos para cada turno utilizando un modelo *SentenceTransformer*.
4. Guardamos los embeddings generados a partir de `all-mpnet-base-v2` y `all-MiniLM-L6-v2` y los IDs asociados en formato `.npy` para reutilizarlos en otras notebooks (indexación y evaluación de ANN).

## 1️⃣ Descarga y exploración del dataset MultiWOZ 2.2

En primer lugar descargamos el dataset desde Kaggle utilizando la librería `kagglehub`.  
Luego inspeccionamos la estructura de directorios y comprobamos que los tres splits (`train`, `dev`, `test`) contengan los archivos JSON esperados.

Cada archivo JSON incluye una **lista de diálogos**, y cada diálogo está compuesto por múltiples turnos (*utterances*) intercambiados entre usuario y sistema.

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("taejinwoo/multiwoz-22")

print("Path to dataset files:", path)

100%|██████████| 14.8M/14.8M [00:00<00:00, 60.9MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/taejinwoo/multiwoz-22/versions/1


In [ ]:
import os

path_train = f"{path}/MultiWOZ_2.2/train/"

train_files = os.listdir(path_train)

print(train_files)

['dialogues_013.json', 'dialogues_006.json', 'dialogues_010.json', 'dialogues_015.json', 'dialogues_001.json', 'dialogues_003.json', 'dialogues_007.json', 'dialogues_012.json', 'dialogues_005.json', 'dialogues_016.json', 'dialogues_014.json', 'dialogues_002.json', 'dialogues_004.json', 'dialogues_017.json', 'dialogues_008.json', 'dialogues_011.json', 'dialogues_009.json']


---

## 2️⃣ Construcción del DataFrame unificado de turnos

A continuación recorremos los archivos de `train`, `dev` y `test` y, para cada diálogo, extraemos sus turnos uno por uno.  
Con esta información construimos un único DataFrame con la siguiente estructura:

- `dialogue_id`: identificador del diálogo (derivado del nombre del archivo).
- `split`: partición a la que pertenece (`train`, `dev` o `test`).
- `turn_id`: índice del turno dentro del diálogo.
- `speaker`: emisor del turno (`USER` o `SYSTEM`).
- `utterance`: texto del turno.

Este DataFrame representa el **corpus plano de turnos** sobre el que calcularemos los embeddings.

In [ ]:
import os
import json
import pandas as pd

# Ruta base (ya creada por KaggleHub)
base_path = f"{path}/MultiWOZ_2.2"

splits = ["train", "dev", "test"]
rows = []

for split in splits:
    split_path = os.path.join(base_path, split)

    for fname in os.listdir(split_path):
        if not fname.endswith(".json"):
            continue

        with open(os.path.join(split_path, fname), "r") as f:
            dialogues = json.load(f)

        # Cada archivo contiene una lista de diálogos
        for dialog in dialogues:
            did = dialog["dialogue_id"]

            # Extraer turnos
            for turn_id, turn in enumerate(dialog["turns"]):
                rows.append({
                    "dialogue_id": did,
                    "split": split,
                    "turn_id": turn_id,
                    "speaker": turn["speaker"],
                    "utterance": turn["utterance"]
                })

df = pd.DataFrame(rows)

In [ ]:
print("Total de turnos cargados:", len(df))

df.head()

Total de turnos cargados: 143044


,dialogue_id,split,turn_id,speaker,utterance
0,SNG0693.json,train,0,USER,I would like to find a restaurant named Tandoo...
1,SNG0693.json,train,1,SYSTEM,I do. It's a very nice Indian restaurant on th...
2,SNG0693.json,train,2,USER,Can you give me the postcode for it please?
3,SNG0693.json,train,3,SYSTEM,The postcode is cb43le. May I help you with an...
4,SNG0693.json,train,4,USER,"That will be all, thanks. Goodbye!"


Persistimos y descargarmos el dataframe para utilizarlo en el resto del pipeline:

In [ ]:
# Guardar DataFrame completo
df.to_pickle("multiwoz_turns.pkl")

print("Archivo guardado: multiwoz_turns.pkl")
print("Shape:", df.shape)

In [ ]:
from google.colab import files

files.download("multiwoz_turns.pkl")

---

## 3️⃣ Generación y guardado de embeddings

Utilizamos un modelo pre-entrenado de `sentence-transformers` para obtener un **embedding d-dimensional** por cada `utterance` del DataFrame.  
El modelo se ejecuta en GPU cuando está disponible (`cuda`) y en caso contrario cae automáticamente a CPU.

El resultado es una matriz `N × D` donde:

- `N` es el número total de turnos del corpus.
- `D` es la dimensión del embedding (por ejemplo, 768).

Finalmente, convertimos los embeddings a `float32`, generamos un vector de IDs enteros (`0, 1, ..., N-1`) y guardamos ambos en disco como:

- `embeddings.npy`: matriz de embeddings.
- `ids.npy`: IDs asociados a cada vector.

Estos archivos serán la entrada de la siguiente notebook, donde construiremos distintos índices FAISS y evaluaremos algoritmos de búsqueda aproximada.


In [ ]:
!pip install -q sentence-transformers

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np
import torch

model_names = {
    "mpnet": "sentence-transformers/all-mpnet-base-v2",
    "minilm": "sentence-transformers/all-MiniLM-L6-v2",
}

if torch.cuda.is_available():
    device = "cuda"
    print("✅ GPU detectada — usando CUDA.")
else:
    device = "cpu"
    print("⚠️ No hay GPU — usando CPU.")

In [ ]:
sentences = df["utterance"].tolist()

len(sentences)

In [ ]:
embeddings_dict = {}

for short_name, model_name in model_names.items():
    print(f"\n🔹 Generando embeddings con {model_name}")
    model = SentenceTransformer(model_name)

    emb = model.encode(
        sentences,
        batch_size=64,
        show_progress_bar=True,
        convert_to_numpy=True,
        device=device
    )

    embeddings_dict[short_name] = emb.astype("float32")
    print(f"Forma de los embeddings ({short_name}):", embeddings_dict[short_name].shape)

In [ ]:
for short_name, emb in embeddings_dict.items():
    print(f"{short_name}: {emb.shape}")

### 4️⃣ Guardado de embeddings y metadatos

Para usar FAISS de forma eficiente, guardamos:

- `embeddings_mpnet.npy`: matriz `N × D` generada con `all-mpnet-base-v2`
- `embeddings_minilm.npy`: matriz `N × D` generada con `all-MiniLM-L6-v2`
- `ids.npy`: arreglo con los IDs de cada vector.

In [ ]:
# ID únicos para cada vector
ids = np.arange(len(sentences), dtype="int64")
np.save("ids.npy", ids)

# Guardar embeddings por modelo
np.save("embeddings_mpnet.npy", embeddings_dict["mpnet"])
np.save("embeddings_minilm.npy", embeddings_dict["minilm"])

embeddings_dict["mpnet"].shape, embeddings_dict["minilm"].shape, ids.shape

Los descargamos a la PC para seguir con el pipeline de trabajo:

In [ ]:
files.download("embeddings_mpnet.npy")
files.download("embeddings_minilm.npy")

files.download("ids.npy")